In [1]:
%cd ../

/home/student/pubtrends


In [2]:
!pwd

/home/student/pubtrends


In [59]:
import logging

import numpy as np
import pandas as pd

from pysrc.papers.db.loader import Loader
from pysrc.papers.db.postgres_connector import PostgresConnector
from pysrc.papers.db.postgres_utils import preprocess_search_query_for_postgres, \
    process_bibliographic_coupling_postgres, process_cocitations_postgres, no_stemming_filter
from pysrc.papers.utils import crc32, SORT_MOST_CITED, SORT_MOST_RECENT, preprocess_doi, \
    preprocess_search_title

logger = logging.getLogger(__name__)

import logging
import html
import pandas as pd
import numpy as np
import networkx as nx
import random
import hashlib
from tqdm.auto import tqdm

from pysrc.prediction.ss_arxiv_loader import SSArxivLoader
from pysrc.prediction.ss_pubmed_loader import SSPubmedLoader
from pysrc.papers.db.pm_postgres_loader import PubmedPostgresLoader
from pysrc.prediction.predict_analyzer import PredictAnalyzer
from pysrc.papers.config import PubtrendsConfig
from pysrc.papers.db.ss_postgres_loader import SemanticScholarPostgresLoader
from pysrc.papers.db.postgres_connector import PostgresConnector
from collections import defaultdict



class CustomLoader(SemanticScholarPostgresLoader):
    def __init__(self, config):
        super(CustomLoader, self).__init__(config)

    def load_func(self, limit=100):
        self.check_connection()
        if limit is None:
            query = '''SELECT ssid, aux::json->'authors' FROM sspublications'''
        else:
            query = f'''
                    SELECT ssid, aux::json->'authors' FROM sspublications LIMIT {limit}
            '''
        result = defaultdict(list)
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            for item in cursor:
                ind, names = item
                for i, el in enumerate(names):
                    result[el['name']].append((ind, int(i == 0)))
            return result
    
    def custom_query(self, query):
        self.check_connection()
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            return cursor.fetchall()
    
    def create_subsample(self, threshold=0.01, seed=42):
        self.check_connection()
        random.seed(seed)
        ssids, crc32ids = [], []
        query = '''select ssid, crc32id from sspublications'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            for item in cursor:
                if random.random() < threshold:
                    ss, crc = item
                    ssids.append(ss)
                    crc32ids.append(crc)
            return ssids, crc32ids


class CustomWriter(PostgresConnector):

    def __init__(self, config):
        super(CustomWriter, self).__init__(config, readonly=False)
        
    def insert_table_publications(self, ids):
        self.check_connection()
        query = f'''CREATE TABLE IF NOT EXISTS sspublications_sample AS 
            (SELECT * FROM sspublications WHERE ssid IN ({', '.join(map(lambda x: "'" + str(x) + "'", ids))}))'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            self.postgres_connection.commit()
        
    def insert_table_citations(self, ids):
        self.check_connection()
        query = f'''CREATE TABLE IF NOT EXISTS sscit_sample AS 
            (SELECT * FROM sscitations WHERE crc32id_in IN ({', '.join(map(lambda x: "'" + str(x) + "'", ids))}));
            create index if not exists sscit_crc32id_in on sscit_sample (crc32id_in);'''
        with self.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            self.postgres_connection.commit()
    
    def execute_custom_query(self, query):
        try:
            self.check_connection()
            with self.postgres_connection.cursor() as cursor:
                cursor.execute(query)
                self.postgres_connection.commit()
        except Exception as e:
            print(e)


In [60]:
config = PubtrendsConfig(test=False)
loader = CustomLoader(config)
writer = CustomWriter(config)

### Selecting 1% subsample for experiments

In [17]:
%%time
ssids, crcids = loader.create_subsample(threshold=0.01)

CPU times: user 47.6 s, sys: 7.68 s, total: 55.3 s
Wall time: 4min 38s


In [6]:
%%time
writer.insert_table_publications(ssids)

CPU times: user 390 ms, sys: 63.1 ms, total: 453 ms
Wall time: 9min 32s


In [18]:
%%time
writer.insert_table_citations(crcids)

CPU times: user 47.1 ms, sys: 9.37 ms, total: 56.4 ms
Wall time: 28min 47s


### Features calculating

### Table schema for authors - papers

(author_id, paper_id, year)

#### Productivity

SELECT t.author_id, avg(t.count) from (select author_id, count(year) from authors_papers group by (author_id, year)) as t group by t.author_id;

### Create table for authors_papers

In [22]:
import string
import zlib


punc = set(string.punctuation)


def process_name(st):
    return ''.join([item for item in st if item not in punc]).strip()


def select_authors():
    data = defaultdict(list)
    loader.check_connection()
    query = '''SELECT ssid, crc32id, year, aux::json->'authors' FROM sspublications_sample'''
    with loader.postgres_connection.cursor() as cursor:
        cursor.execute(query)
        for item in cursor:
            ind, crc, year, names = item
            for i, el in enumerate(names):
                data[process_name(el['name'])].append((ind, crc, year, int(i == 0)))
        del data['']
        return data
    

def process_data(data):
    results = []
    tmp = {True: 'TRUE', False: 'FALSE'}
    for item in data:
        if item:
            for ind, crc, year, is_main in data[item]:
                results.append(f'''({zlib.crc32(item.encode())}, '{ind}', {crc}, {year if year is not None else "NULL"}, '{item[:100]}', {tmp[is_main]})''')
    return results

def insert_authors_papers_data(data):
    try:
        writer.check_connection()
        query_init = '''CREATE TABLE IF NOT EXISTS authors_papers (
                            author_id bigint not null, 
                            ssid varchar(40) not null,
                            crc32id_paper integer not null,
                            year integer,
                            author_name varchar(100) not null, 
                            is_main boolean);
                        create index if not exists authors_papers_crcid_author_year on authors_papers (author_id, year);'''
        
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query_init)
            writer.postgres_connection.commit()
            
        with writer.postgres_connection.cursor() as cursor:
            query_insert = f'''insert into authors_papers(author_id, ssid, crc32id_paper, year, author_name, is_main) values {','.join(data)}'''
            cursor.execute(query_insert)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)


    

In [23]:
%%time
df = select_authors()
data = process_data(df)
insert_authors_papers_data(data)

CPU times: user 22.5 s, sys: 872 ms, total: 23.4 s
Wall time: 1min 27s


### Create table for authors

In [24]:
def create_authors_table():
    query = '''create table if not exists authors as (select distinct on (author_id) author_id, author_name from authors_papers)'''
    writer.execute_custom_query(query=query)


In [25]:
%%time
create_authors_table()

CPU times: user 1.28 ms, sys: 204 µs, total: 1.48 ms
Wall time: 4.27 s


### Calculate productivity


In [26]:
%%time
query_prod = '''alter table authors add column if not exists productivity decimal;
                    update authors set productivity = t.prod 
                        from (select author_id, cast(count(*) as decimal) / (2021 - min(year) + 1) as prod 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_prod)

CPU times: user 2.09 ms, sys: 0 ns, total: 2.09 ms
Wall time: 10.1 s


### Calculate max and min year of publications

In [27]:
%%time
query_years = '''alter table authors add column if not exists first_year_pub int;
                    alter table authors add column if not exists last_year_pub int;
                    update authors set first_year_pub = t.res 
                        from (select author_id, min(year) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;
                    update authors set last_year_pub = t.res 
                        from (select author_id, max(year) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_years)

CPU times: user 0 ns, sys: 2.11 ms, total: 2.11 ms
Wall time: 18.4 s


### Calculate total papers

In [28]:
%%time
query_papers = '''alter table authors add column if not exists total_papers int;
                    update authors set total_papers = t.res 
                        from (select author_id, count(*) as res 
                            from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_papers)

CPU times: user 1.56 ms, sys: 0 ns, total: 1.56 ms
Wall time: 8.81 s


### Calculate ratio of main authering


In [29]:
%%time
query_is_main = '''alter table authors add column if not exists is_main_ratio decimal;
                    update authors set is_main_ratio = t.res 
                        from (select author_id, cast(count(case when is_main then 1 end) as decimal) / count(*) as res
                        from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_is_main)

CPU times: user 1.68 ms, sys: 0 ns, total: 1.68 ms
Wall time: 10.9 s


### Calculate avg num of coauthors

In [30]:
%%time
query_coauthors = '''alter table authors add column if not exists sociality decimal;
                    update authors set sociality = t.res 
                        from (select author_id, avg(count - 1) as res
                            from (select * from authors_papers left join (select crc32id_paper, count(*)
                                from authors_papers group by crc32id_paper) as f on authors_papers.crc32id_paper=f.crc32id_paper) as tab group by tab.author_id) 
                                as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_coauthors)

CPU times: user 1.61 ms, sys: 207 µs, total: 1.82 ms
Wall time: 12.6 s


### Calculate total num of papers

In [31]:
%%time
query_total_papers = '''alter table authors add column if not exists total_papers int;
                    update authors set total_papers = t.res 
                        from (select author_id, count(*) as res from authors_papers group by author_id) as t where t.author_id = authors.author_id;'''
writer.execute_custom_query(query=query_total_papers)

CPU times: user 1.62 ms, sys: 208 µs, total: 1.83 ms
Wall time: 9.21 s


### Calculcate total num of venues

In [32]:
%%time
q = '''select authors_papers.author_id, ven, jour from authors_papers left join (SELECT crc32id, aux::json->'venue' as ven, aux::json#>'{journal, name}' as jour
                                        FROM sspublications_sample) as t on authors_papers.crc32id_paper = t.crc32id'''
journals_and_venues = loader.custom_query(q)

CPU times: user 14 s, sys: 433 ms, total: 14.4 s
Wall time: 4min 4s


In [33]:
def calc_total_venues(data):
    try:
        authors = {}
        for a_id, ven, jour in data:
            if a_id not in authors:
                authors[a_id] = set()
            if ven:
                authors[a_id].add(ven)
            if jour:
                authors[a_id].add(jour)
        ls = []
        for key in authors:
            ls.append(f'''({key}, {len(authors[key])})''')
        
        query = f'''
                    alter table authors add column if not exists total_venues int;
                    create table if not exists temp_table (a_id bigint, venues int);
                    insert into temp_table values {','.join(ls)};
                    update authors set total_venues = t.venues 
                        from (select a_id, venues from temp_table) as t where t.a_id = authors.author_id;
                    drop table temp_table;
                    '''
        writer.check_connection()
        with writer.postgres_connection.cursor() as cursor:
            cursor.execute(query)
            writer.postgres_connection.commit()
    except Exception as e:
        print(e)
        
        

In [34]:
%%time
calc_total_venues(journals_and_venues)

CPU times: user 5.61 s, sys: 250 ms, total: 5.86 s
Wall time: 24.5 s


### Calculate citations (c2, c5, c_all)

In [35]:
# all
# select crc32id_in, count(crc32id_out) from sscit_sample group by crc32id_in;

# alter table sscit_sample add column if not exists out_year int;
# alter table sscit_sample add column if not exists in_year int;
# update sscit_sample set out_year = t.year from (select crc32id, year from sspublications) as t where t.crc32id = sscit_sample.crc32id_out;
# update sscit_sample set in_year = t.year from (select crc32id, year from sspublications_sample) as t where t.crc32id = sscit_sample.crc32id_in;

# c2
# select crc32id_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;
# alter table sspublications_sample add column if not exists c2 int;
# update sspublications_sample set c2 = t.res from (select crc32id_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) as res from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;

# c5
# select crc32id_in, sum(case when out_year - in_year <= 5 and  out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_all
# select crc32id_in, sum(case when out_year - in_year >= 0 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;


In [5]:
%%time
query_in_year_and_out_year = '''
                alter table sscit_sample add column if not exists out_year int;
                alter table sscit_sample add column if not exists in_year int;
                update sscit_sample set out_year = t.year from (select crc32id, year from sspublications) as t where t.crc32id = sscit_sample.crc32id_out;
                update sscit_sample set in_year = t.year from (select crc32id, year from sspublications_sample) as t where t.crc32id = sscit_sample.crc32id_in;
                '''
writer.execute_custom_query(query=query_in_year_and_out_year)

CPU times: user 10.2 ms, sys: 4.53 ms, total: 14.8 ms
Wall time: 7min 30s


In [6]:
%%time
query_c2 = '''
                alter table sspublications_sample add column if not exists c2 int default 0;
                update sspublications_sample set c2 = t.res 
                    from (select crc32id_in, sum(case when out_year - in_year <= 2 and  out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.crc32id_in) as t
                            where t.crc32id_in = sspublications_sample.crc32id;
                '''
writer.execute_custom_query(query=query_c2)

CPU times: user 3.55 ms, sys: 363 µs, total: 3.91 ms
Wall time: 18.5 s


In [7]:
%%time
query_c5 = '''
                alter table sspublications_sample add column if not exists c5 int default 0;
                update sspublications_sample set c5 = t.res 
                    from (select crc32id_in, sum(case when out_year - in_year <= 5 and  out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.crc32id_in) as t
                            where t.crc32id_in = sspublications_sample.crc32id;
            '''
writer.execute_custom_query(query=query_c5)

CPU times: user 1.79 ms, sys: 227 µs, total: 2.02 ms
Wall time: 14 s


In [9]:
%%time
query_c_all = '''
                alter table sspublications_sample add column if not exists c_all int default 0;
                update sspublications_sample set c_all = t.res 
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 then 1 else 0 end) as res 
                        from sscit_sample group by sscit_sample.crc32id_in) as t
                            where t.crc32id_in = sspublications_sample.crc32id;
                '''
writer.execute_custom_query(query=query_c_all)

CPU times: user 0 ns, sys: 1.83 ms, total: 1.83 ms
Wall time: 12.8 s


### Caclculate mean citations for authors

In [10]:
%%time
query_mean_citations = '''
                        alter table authors add column if not exists mean_citations decimal;
                        update authors set mean_citations = t.res 
                                            from (select f.author_id, avg(f.c_all) as res from (select author_id, c_all 
                                            from authors_papers left join sspublications_sample on authors_papers.crc32id_paper = sspublications_sample.crc32id) as f 
                                            group by f.author_id) as t where t.author_id = authors.author_id;
                        '''
writer.execute_custom_query(query=query_mean_citations)

CPU times: user 1.62 ms, sys: 0 ns, total: 1.62 ms
Wall time: 12.8 s


### Calculate top cited paper for authors

In [11]:
%%time
query_top_citatoons = '''
                        alter table authors add column if not exists top_citations int;
                        update authors set top_citations = t.res 
                            from (select f.author_id, max(f.c_all) as res from (select author_id, c_all 
                            from authors_papers left join sspublications_sample on authors_papers.crc32id_paper = sspublications_sample.crc32id) as f
                            group by f.author_id) as t where t.author_id = authors.author_id;
                        '''
writer.execute_custom_query(query=query_top_citatoons)

CPU times: user 1.76 ms, sys: 0 ns, total: 1.76 ms
Wall time: 10.9 s


### Calculate h-index

In [ ]:
# c_1970
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1970 and in_year <= 1970 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_1980
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1980 and in_year <= 1980 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_1990
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1990 and in_year <= 1990 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2000
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2000 and in_year <= 2000 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2010
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2010 and in_year <= 2010 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

# c_2020
# select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2020 and in_year <= 2020 then 1 else 0 end) from sscit_sample group by sscit_sample.crc32id_in;

In [16]:
%%time 
query_c_1970 = '''
                alter table sspublications_sample add column c_1970 int default 0;
                update sspublications_sample set c_1970 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1970 and in_year <= 1970 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_1970)

CPU times: user 899 µs, sys: 3.35 ms, total: 4.24 ms
Wall time: 12.6 s


In [17]:
%%time 
query_c_1980 = '''
                alter table sspublications_sample add column c_1980 int default 0;
                update sspublications_sample set c_1980 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1980 and in_year <= 1980 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_1980)

CPU times: user 1.76 ms, sys: 234 µs, total: 1.99 ms
Wall time: 13.6 s


In [18]:
%%time 
query_c_1990 = '''
                alter table sspublications_sample add column c_1990 int default 0;
                update sspublications_sample set c_1990 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 1990 and in_year <= 1990 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_1990)

CPU times: user 1.48 ms, sys: 0 ns, total: 1.48 ms
Wall time: 13.3 s


In [24]:
%%time 
query_c_2000 = '''
                alter table sspublications_sample add column c_2000 int default 0;
                update sspublications_sample set c_2000 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2000 and in_year <= 2000 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_2000)

CPU times: user 4.6 ms, sys: 0 ns, total: 4.6 ms
Wall time: 12.8 s


In [25]:
%%time 
query_c_2010 = '''
                alter table sspublications_sample add column c_2010 int default 0;
                update sspublications_sample set c_2010 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2010 and in_year <= 2010 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_2010)

CPU times: user 1.75 ms, sys: 0 ns, total: 1.75 ms
Wall time: 12.7 s


In [26]:
%%time 
query_c_2020 = '''
                alter table sspublications_sample add column c_2020 int default 0;
                update sspublications_sample set c_2020 = t.res
                    from (select crc32id_in, sum(case when out_year - in_year >= 0 and out_year <= 2020 and in_year <= 2020 then 1 else 0 end) as res
                    from sscit_sample group by sscit_sample.crc32id_in) as t where t.crc32id_in = sspublications_sample.crc32id;
'''
writer.execute_custom_query(query=query_c_2020)

CPU times: user 1.71 ms, sys: 0 ns, total: 1.71 ms
Wall time: 13.9 s


In [29]:
def hindex(citations):
    ls = sorted(citations, reverse=True)
    for h in range(len(ls)):
        if h + 1 > ls[h]:
            return h
    return len(ls)

In [49]:
%%time
query = '''select author_id from authors_papers left join sspublications_sample on authors_papers.crc32id_paper = sspublications_sample.crc32id'''
ls = loader.custom_query(query) 

CPU times: user 761 ms, sys: 118 ms, total: 879 ms
Wall time: 3.47 s


In [50]:
len(ls)

4294580

In [44]:
ll = loader.custom_query('''select author_id from authors_papers;''')

In [45]:
len(ll)

4287846

In [54]:
from collections import Counter


c1 = Counter(ls)
c2 = Counter(ll)
s = set()
for item in c1:
    if c1[item] != c2[item]:
        s.add((item, c1[item], c2[item]))

In [55]:
list(s)[:10]

[((3612005501,), 2, 1),
 ((3812315064,), 4, 2),
 ((1221578854,), 2, 1),
 ((1616422563,), 2, 1),
 ((14882713,), 2, 1),
 ((3468535478,), 8, 6),
 ((1166534910,), 4, 2),
 ((3101503964,), 5, 3),
 ((1692554633,), 4, 2),
 ((2813311664,), 3, 2)]

In [46]:
set([item[0] for item in ls]) - set([item[0] for item in ll])

set()

### Calculate prestiges for venues

In [21]:
q = '''select crc32id_out from sscit_sample;'''
ls = loader.custom_query(q)

In [45]:
q = f'''select year from sspublications where crc32id in ({', '.join(map(lambda x: str(x[0]), ls))})'''

In [46]:
%%time
ll = loader.custom_query(q)

InternalError_: invalid memory alloc request size 1073741824


In [30]:
ll[:10]

[(-2147358840, 2013),
 (-2147096883, 2018),
 (-2146724546, 2019),
 (-2146469899, 2007),
 (-2146148768, 2020),
 (-2145351624, 2015),
 (-2145059682, 2009),
 (-2145004304, 2014),
 (-2144593116, None),
 (-2144389263, 2017)]

In [48]:
len(ls)

13077703

In [15]:
ls = loader.custom_query('''select crc32id_in from sscit_sample;''')

In [16]:
s = set([item[0] for item in ls])
len(s)

661235

In [61]:
ls_crc = loader.custom_query('''select crc32id from sspublications_sample;''')
s = set([item[0] for item in ls_crc])
len(s)

1584844

In [62]:
cc = Counter(ls_crc)
cc.most_common(10)

[((-1943044845,), 3),
 ((-2142109262,), 2),
 ((-2141656025,), 2),
 ((-2141582078,), 2),
 ((-2139624107,), 2),
 ((-2134292316,), 2),
 ((-2134109782,), 2),
 ((-2126981967,), 2),
 ((-2125613728,), 2),
 ((-2122298027,), 2)]

In [20]:
len(set(crcids))

1584867

In [21]:
len(ssids)

1585169

In [ ]:
 select count(*) from (select f.author_id, avg(f.c_all) as res from (select author_id, c_all from authors_papers left join sspublications_sample on authors_papers.crc32id_paper = sspublications
_sample.crc32id) as f group by f.author_id) as tab where tab.res is null;